# xG & Shooting Profile Analysis
### Julie Landrevie — Football Data Analyst
---
**Objectif de ce notebook :**  
Analyser la qualité des tirs en utilisant les données StatsBomb open data.  
On part de données brutes pour arriver à des insights tactiques actionnables.

**Dataset :** La Liga 2020/2021 — StatsBomb Open Data  
**Métriques clés :** xG (expected goals), overperformance, profil de tir, zones de danger

**Pipeline du notebook :**
1. 📦 Setup & chargement des données
2. 🔍 Exploration rapide
3. 🗺️ Shot Map — visualiser les tirs sur le terrain
4. 👤 Profil tireur — xG vs buts réels
5. 🎯 Overperformance — qui surperforme ou sous-performe ?
6. 🗺️ Heatmap des zones de danger
7. 📊 Distribution xG — qualité des occasions
8. ⏱️ Timeline xG — évolution match par match
9. 🔎 Analyse ciblée — deep dive sur un joueur
10. 💡 Conclusions & insights

---
## 1. Setup & Imports

On importe nos trois modules maison :
- `data_loader` : récupération et nettoyage des données StatsBomb
- `metrics` : calcul des indicateurs xG
- `viz` : toutes les visualisations

In [1]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Modules du projet
import sys
sys.path.insert(0, "..")

from src.data_loader import (
    list_competitions,
    load_matches,
    load_shots_from_match,
    load_shots_from_competition,
    filter_shots,
    quick_summary,
)
from src.metrics import (
    player_shooting_profile,
    team_shooting_profile,
    xg_by_zone,
    xg_distribution,
    xg_by_time_period,
    top_shooters,
    compare_players,
)
from src.viz import (
    shot_map,
    player_xg_bar,
    overperformance_chart,
    xg_distribution_plot,
    xg_zone_heatmap,
    xg_timeline,
)

print("✅ Imports OK")

✅ Imports OK


---
## 2. Explorer les données disponibles

StatsBomb met à disposition plusieurs compétitions gratuitement.  
Voyons ce qui est accessible.

In [2]:
# Toutes les compétitions StatsBomb open data
comps = list_competitions()
print(f"{len(comps)} compétitions disponibles\n")
comps.head(20)

75 compétitions disponibles



,competition_id,season_id,competition_name,season_name
0,9,27,1. Bundesliga,2015/2016
1,9,281,1. Bundesliga,2023/2024
2,1267,107,African Cup of Nations,2023
3,16,276,Champions League,1970/1971
4,16,71,Champions League,1971/1972
5,16,277,Champions League,1972/1973
6,16,76,Champions League,1999/2000
7,16,44,Champions League,2003/2004
8,16,37,Champions League,2004/2005
9,16,39,Champions League,2006/2007


In [3]:
# On travaille sur la La Liga 2020/2021
# → compétition_id=11, season_id=90
COMPETITION_ID = 11
SEASON_ID      = 90

matches = load_matches(COMPETITION_ID, SEASON_ID)
print(f"{len(matches)} matchs chargés\n")
matches[["match_date", "home_team", "away_team", "home_score", "away_score"]].head(10)

35 matchs chargés



,match_date,home_team,away_team,home_score,away_score
7,2020-09-27,Barcelona,Villarreal,4,0
8,2020-10-01,Celta Vigo,Barcelona,0,3
11,2020-10-04,Barcelona,Sevilla,1,1
12,2020-10-17,Getafe,Barcelona,1,0
9,2020-10-24,Barcelona,Real Madrid,1,3
0,2020-10-31,Deportivo Alavés,Barcelona,1,1
34,2020-11-07,Barcelona,Real Betis,5,2
13,2020-11-21,Atlético Madrid,Barcelona,1,0
30,2020-11-29,Barcelona,Osasuna,4,0
24,2020-12-05,Cádiz,Barcelona,2,1


---
## 3. Chargement des tirs — saison complète

On charge TOUS les tirs de la saison en une seule commande.  
Les données sont mises en cache pour les prochaines exécutions.

In [4]:
# Chargement de tous les tirs de la saison
# Première exécution : ~2 min | Suivantes : instantané (cache CSV)
shots = load_shots_from_competition(
    competition_id=COMPETITION_ID,
    season_id=SEASON_ID,
    use_cache=True,
    verbose=True
)

📂 Cache trouvé — chargement depuis data/cache/shots_11_90.csv


In [5]:
# Vérification rapide du dataset
quick_summary(shots)

📊 RÉSUMÉ DES DONNÉES
  Matchs analysés   : 35
  Équipes           : 19
  Joueurs tireurs   : 185
  Tirs totaux       : 839
  Buts réels        : 111
  xG total          : 107.4
  Conversion réelle : 13.2%
  Conversion xG     : 1.03 (1.0 = normal)

  Répartition par résultat :
outcome
Off T               256
Saved               227
Blocked             203
Goal                111
Wayward              20
Post                 15
Saved Off Target      4
Saved to Post         3

  Répartition par partie du corps :
body_part
Left Foot     400
Right Foot    329
Head          105
Other           5


In [6]:
# Aperçu des données brutes
shots[[
    "player", "team", "minute", "xg", "outcome",
    "body_part", "technique", "x", "y", "is_goal"
]].head(10)

,player,team,minute,xg,outcome,body_part,technique,x,y,is_goal
0,Philippe Coutinho Correia,Barcelona,5,0.041129,Blocked,Right Foot,Normal,96.5,32.0,False
1,Anssumane Fati,Barcelona,14,0.083005,Goal,Right Foot,Normal,107.2,32.3,True
2,Sergi Roberto Carnicer,Barcelona,16,0.050400,Off T,Left Foot,Half Volley,106.5,41.3,False
3,Anssumane Fati,Barcelona,18,0.442780,Goal,Right Foot,Normal,109.0,32.7,True
4,Lionel Andrés Messi Cuccittini,Barcelona,25,0.201636,Blocked,Left Foot,Normal,114.0,33.9,False
5,Jordi Alba Ramos,Barcelona,25,0.013961,Saved,Left Foot,Normal,118.5,28.9,False
6,Lionel Andrés Messi Cuccittini,Barcelona,34,0.783500,Goal,Left Foot,Normal,108.0,39.8,True
7,Francis Joseph Coquelin,Villarreal,38,0.025156,Blocked,Right Foot,Normal,102.8,45.5,False
8,Philippe Coutinho Correia,Barcelona,39,0.182982,Saved,Right Foot,Volley,112.4,35.4,False
9,Gerard Moreno Balaguero,Villarreal,44,0.086089,Blocked,Left Foot,Normal,104.3,34.8,False


---
## 4. Shot Map — Visualiser les tirs sur le terrain

**Lecture de la shot map :**
- Chaque point = un tir
- Taille du point = valeur xG (plus gros = plus dangereux)
- 🟡 Or = but marqué
- ⬜ Contour blanc = tir cadré (Goal ou Saved)
- ⬛ Gris = tir non cadré

> **Insight attendu :** Les plus gros points (xG élevé) devraient se concentrer 
> dans l'axe central, proches du but. Les tirs de l'extérieur de la surface auront 
> de petits points malgré leur nombre.

In [7]:
# Shot Map de tout Barcelona sur la saison
fig = shot_map(
    shots,
    title="La Liga 2020/21 — Shot Map Barcelona",
    team="Barcelona"
)
plt.show()

In [8]:
# Comparaison : shot map d'un adversaire
fig = shot_map(
    shots,
    title="La Liga 2020/21 — Shot Map Atlético Madrid",
    team="Atlético Madrid"
)
plt.show()

---
## 5. Profil tireur — xG vs Buts réels

**Métriques calculées pour chaque joueur :**
- `xg` : xG total = somme de la probabilité de but de chaque tir
- `goals` : buts réels marqués
- `overperformance` = buts - xG → mesure l'efficacité du finisseur
- `xg_per_shot` : qualité moyenne des occasions créées
- `on_target_rate` : % de tirs cadrés
- `big_chances` : nombre d'occasions à xG ≥ 0.30

In [9]:
# Calcul du profil pour tous les joueurs de la saison
profile = player_shooting_profile(shots)

print(f"✅ Profil calculé pour {len(profile)} joueurs\n")
profile[[
    "player", "team", "shots", "goals", "xg",
    "overperformance", "xg_per_shot", "on_target_rate", "big_chances"
]].head(15)

✅ Profil calculé pour 185 joueurs



,player,team,shots,goals,xg,overperformance,xg_per_shot,on_target_rate,big_chances
0,Lionel Andrés Messi Cuccittini,Barcelona,195,30,22.87,7.13,0.117,45.6,17
1,Antoine Griezmann,Barcelona,64,12,12.55,-0.55,0.196,45.3,12
2,Ousmane Dembélé,Barcelona,47,5,4.54,0.46,0.097,34.0,2
3,Martin Braithwaite Christensen,Barcelona,22,2,4.35,-2.35,0.198,31.8,5
4,Frenkie de Jong,Barcelona,15,2,4.21,-2.21,0.281,26.7,4
5,Philippe Coutinho Correia,Barcelona,27,2,3.69,-1.69,0.137,33.3,4
6,Pedro González López,Barcelona,22,3,3.64,-0.64,0.165,27.3,5
7,Francisco António Machado Mota de Castro Trincão,Barcelona,20,3,2.39,0.61,0.120,50.0,3
8,Rafael Mir Vicente,Huesca,6,1,2.19,-1.19,0.366,50.0,2
9,Jordi Alba Ramos,Barcelona,24,3,1.96,1.04,0.082,45.8,1


In [10]:
# Graphique xG vs Buts — Top 15 joueurs par xG
fig = player_xg_bar(
    profile,
    top_n=15,
    metric="xg",
    title="La Liga 2020/21 — xG vs Buts (Top 15)"
)
plt.show()

In [11]:
# Top 10 : meilleurs tireurs par xG total
print("🏆 TOP 10 — xG Total")
print("="*60)
top_xg = top_shooters(shots, metric="xg", top_n=10)
print(top_xg[["player", "team", "shots", "goals", "xg", "xg_per_shot"]].to_string(index=False))

🏆 TOP 10 — xG Total
                                          player      team  shots  goals    xg  xg_per_shot
                  Lionel Andrés Messi Cuccittini Barcelona    195     30 22.87        0.117
                               Antoine Griezmann Barcelona     64     12 12.55        0.196
                                 Ousmane Dembélé Barcelona     47      5  4.54        0.097
                  Martin Braithwaite Christensen Barcelona     22      2  4.35        0.198
                                 Frenkie de Jong Barcelona     15      2  4.21        0.281
                       Philippe Coutinho Correia Barcelona     27      2  3.69        0.137
                            Pedro González López Barcelona     22      3  3.64        0.165
Francisco António Machado Mota de Castro Trincão Barcelona     20      3  2.39        0.120
                              Rafael Mir Vicente    Huesca      6      1  2.19        0.366
                                Jordi Alba Ramos Barcelona  

In [12]:
# Top 10 : meilleurs finisseurs (overperformance)
print("\n⚡ TOP 10 — Overperformance (Buts − xG)")
print("="*60)
top_over = top_shooters(shots, metric="overperformance", top_n=10, min_shots=5)
print(top_over[["player", "team", "shots", "goals", "xg", "overperformance"]].to_string(index=False))


⚡ TOP 10 — Overperformance (Buts − xG)
                                          player            team  shots  goals    xg  overperformance
                  Lionel Andrés Messi Cuccittini       Barcelona    195     30 22.87             7.13
                                  Anssumane Fati       Barcelona     14      4  1.62             2.38
                           Óscar Mingueza García       Barcelona      9      2  0.80             1.20
                                Jordi Alba Ramos       Barcelona     24      3  1.96             1.04
                       José Luis Morales Nogales      Levante UD      5      1  0.22             0.78
                       Yannick Ferreira Carrasco Atlético Madrid      6      1  0.35             0.65
Francisco António Machado Mota de Castro Trincão       Barcelona     20      3  2.39             0.61
                          Sergi Roberto Carnicer       Barcelona      5      1  0.43             0.57
                                 Ousmane D

In [13]:
# Top 10 : meilleurs sélecteurs de tir (xG/tir)
# → joueurs qui choisissent bien leurs tirs (peu mais qualité)
print("\n🎯 TOP 10 — xG par tir (min. 10 tirs)")
print("="*60)
top_quality = top_shooters(shots, metric="xg_per_shot", top_n=10, min_shots=10)
print(top_quality[["player", "team", "shots", "goals", "xg", "xg_per_shot", "on_target_rate"]].to_string(index=False))


🎯 TOP 10 — xG par tir (min. 10 tirs)
                                          player      team  shots  goals    xg  xg_per_shot  on_target_rate
                                 Frenkie de Jong Barcelona     15      2  4.21        0.281            26.7
                  Martin Braithwaite Christensen Barcelona     22      2  4.35        0.198            31.8
                               Antoine Griezmann Barcelona     64     12 12.55        0.196            45.3
                 Ronald Federico Araújo da Silva Barcelona     10      2  1.79        0.179            30.0
                            Pedro González López Barcelona     22      3  3.64        0.165            27.3
                       Philippe Coutinho Correia Barcelona     27      2  3.69        0.137            33.3
                                 Clément Lenglet Barcelona     10      1  1.34        0.134            50.0
                                    Sergino Dest Barcelona     14      2  1.77        0.126       

---
## 6. Overperformance — Qui surperforme ou sous-performe ?

**Overperformance = Buts réels − xG attendu**

- **Positif (vert)** : le joueur marque plus que prévu → finisseur efficace, ou période de chance
- **Négatif (rouge)** : le joueur marque moins que prévu → manque d'efficacité, ou malchance

> ⚠️ Sur une saison entière, un joueur très positif est probablement un grand finisseur.  
> Sur 5 matchs, c'est plus probablement de la variance statistique.

In [14]:
# Lollipop chart overperformance — Top 15
fig = overperformance_chart(
    profile,
    top_n=15,
    title="La Liga 2020/21 — Overperformance xG (Top 15 par xG)"
)
plt.show()

In [15]:
# Focus : joueurs les plus sous-performants (malchanceux ou inefficaces)
sous_perf = profile[profile["shots"] >= 10].sort_values("overperformance").head(10)
print("\n📉 Joueurs les plus sous-performants (min. 10 tirs)")
print(sous_perf[["player", "team", "shots", "goals", "xg", "overperformance"]].to_string(index=False))


📉 Joueurs les plus sous-performants (min. 10 tirs)
                         player      team  shots  goals    xg  overperformance
 Martin Braithwaite Christensen Barcelona     22      2  4.35            -2.35
                Frenkie de Jong Barcelona     15      2  4.21            -2.21
      Philippe Coutinho Correia Barcelona     27      2  3.69            -1.69
                 Miralem Pjanić Barcelona     14      0  0.97            -0.97
           Pedro González López Barcelona     22      3  3.64            -0.64
              Antoine Griezmann Barcelona     64     12 12.55            -0.55
                Clément Lenglet Barcelona     10      1  1.34            -0.34
       Moriba Kourouma Kourouma Barcelona     13      1  1.15            -0.15
Ronald Federico Araújo da Silva Barcelona     10      2  1.79             0.21
                   Sergino Dest Barcelona     14      2  1.77             0.23


---
## 7. Heatmap des zones de danger

**Lecture de la heatmap :**  
Chaque zone est colorée selon le xG cumulé : plus c'est chaud, plus les tirs 
qui en viennent sont collectivement dangereux.

> **Insight attendu :** Le centre de la surface (en face du but) devrait être 
> la zone la plus rouge. Un pic sur le côté révèle une équipe qui tire beaucoup 
> de l'aile ou exploite les centres.

In [16]:
# Heatmap — tous les tirs de la saison
fig = xg_zone_heatmap(
    shots,
    title="La Liga 2020/21 — Heatmap xG (toutes équipes)"
)
plt.show()

In [17]:
# Comparaison Barcelona vs adversaires
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
plt.close()

fig = xg_zone_heatmap(shots, title="Barcelona — Heatmap xG", team="Barcelona")
plt.show()
fig = xg_zone_heatmap(shots, title="Atlético Madrid — Heatmap xG", team="Atlético")
plt.show()

In [18]:
# Tableau des stats par zone
print("📊 xG par zone de tir")
print("="*70)
zones = xg_by_zone(shots)
print(zones.to_string(index=False))

📊 xG par zone de tir
                 zone_name  shots  goals  xg_total  xg_mean  conversion
        Penalty / but vide     26     20     21.19    0.815        76.9
Surface centrale (6 yards)    507     79     74.71    0.147        15.6
Entrée de surface centrale    166     10      7.17    0.043         6.0
            Surface droite     41      1      1.72    0.042         2.4
            Surface gauche     42      0      1.33    0.032         0.0
  Entrée de surface droite     25      0      0.62    0.025         0.0
  Entrée de surface gauche     27      1      0.66    0.024         3.7
    Hors surface lointaine      5      0      0.02    0.005         0.0


---
## 8. Distribution xG — Qualité des occasions créées

**Que révèle cet histogramme ?**  
- Pic à gauche (beaucoup de faibles xG) → équipe qui tire de loin, peu d'entrées en surface
- Pic à droite (beaucoup de hauts xG) → équipe qui crée des occasions en face du but
- Barre or (buts) dans les hautes valeurs → finisseur efficace sur les grosses occasions

In [19]:
# Distribution xG — tous les tirs
fig = xg_distribution_plot(
    shots,
    title="La Liga 2020/21 — Distribution xG (toutes équipes)"
)
plt.show()

In [20]:
# Distribution xG — focus Barcelona
fig = xg_distribution_plot(
    shots,
    title="La Liga 2020/21 — Distribution xG Barcelona",
    team="Barcelona"
)
plt.show()

In [21]:
# Table détaillée : distribution par tranche
dist = xg_distribution(shots)
print("Distribution des tirs par tranche xG :")
print(dist[dist["shots"] > 0].to_string(index=False))

Distribution des tirs par tranche xG :
 xg_range  shots  goals  xg_total  conversion
0.00–0.10    556     28     27.28         5.0
0.10–0.20    149     21     20.88        14.1
0.20–0.30     48     14     11.96        29.2
0.30–0.40     24     11      8.25        45.8
0.40–0.50     21     11      9.28        52.4
0.50–0.60     13      5      7.21        38.5
0.60–0.70      2      1      1.37        50.0
0.70–0.80     18     14     14.04        77.8
0.80–0.90      5      4      4.30        80.0
0.90–1.00      3      2      2.85        66.7


---
## 9. Timeline xG — Évolution du danger au fil du match

**Lecture de la timeline :**
- L'axe x = la minute du match
- L'axe y = le xG cumulé
- Un saut vertical = un tir dangereux
- ⭐ = un but marqué
- Mi-temps = ligne pointillée à la 45e

> **Insight tactique :** Si une équipe accumule du xG très rapidement en début de match, 
> elle presse haut et crée des occasions tôt. Un xG qui monte surtout après la 70e 
> révèle une équipe qui pousse en fin de match.

In [22]:
# Timeline xG sur un match spécifique — El Clasico
el_clasico = matches[
    (matches["home_team"].str.contains("Barcelona")) &
    (matches["away_team"].str.contains("Real Madrid"))
].iloc[0]

print(f"Match sélectionné : {el_clasico['home_team']} vs {el_clasico['away_team']} — {el_clasico['match_date']}")
print(f"Score : {el_clasico['home_score']} - {el_clasico['away_score']}")

shots_clasico = load_shots_from_match(el_clasico["match_id"])

fig = xg_timeline(
    shots_clasico,
    home_team=el_clasico["home_team"],
    away_team=el_clasico["away_team"],
    title=f"El Clásico {el_clasico['match_date']} — Timeline xG"
)
plt.show()

Match sélectionné : Barcelona vs Real Madrid — 2020-10-24
Score : 1 - 3


In [23]:
# Évolution xG par tranche de 15 min (saison entière)
tempo = xg_by_time_period(shots)
print("\nxG par tranche de 15 minutes (toutes équipes, saison entière) :")
print(tempo.to_string(index=False))


xG par tranche de 15 minutes (toutes équipes, saison entière) :
period  shots  goals  xg_total  xg_per_shot
  0–15    114     13     15.24        0.134
 15–30    127     19     16.29        0.128
 30–45    107     17     14.80        0.138
 45–60    153     20     22.82        0.149
 60–75    141     22     17.58        0.125
 75–90    161     18     17.62        0.109
90–105     36      2      3.07        0.085


---
## 10. Deep Dive — Analyse d'un joueur spécifique

On va analyser Messi en détail : shot map, profil, distribution, et comparaison 
avec un autre attaquant de la saison.

In [24]:
# Shot map de Messi
messi_shots = filter_shots(shots, player="Messi")
print(f"Messi — {len(messi_shots)} tirs sur la saison")

fig = shot_map(
    messi_shots,
    title="Lionel Messi — Shot Map La Liga 2020/21"
)
plt.show()

Messi — 195 tirs sur la saison


In [25]:
# Profil détaillé Messi
messi_profile = player_shooting_profile(messi_shots)
print("\n👤 PROFIL MESSI — La Liga 2020/21")
print("="*50)
for col in ["shots", "goals", "xg", "xg_per_shot", "overperformance",
            "on_target_rate", "big_chances", "big_chances_scored",
            "headed_shots", "left_foot_shots", "right_foot_shots"]:
    print(f"  {col:25s}: {messi_profile.iloc[0][col]}")


👤 PROFIL MESSI — La Liga 2020/21
  shots                    : 195
  goals                    : 30
  xg                       : 22.87
  xg_per_shot              : 0.117
  overperformance          : 7.13
  on_target_rate           : 45.6
  big_chances              : 17
  big_chances_scored       : 10
  headed_shots             : 5
  left_foot_shots          : 172
  right_foot_shots         : 17


In [26]:
# Distribution xG de Messi
fig = xg_distribution_plot(
    messi_shots,
    title="Lionel Messi — Distribution xG"
)
plt.show()

In [27]:
# Comparaison Messi vs un autre attaquant
# Trouver des attaquants avec au moins 15 tirs
print("Joueurs avec 15+ tirs sur la saison :")
top = profile[profile["shots"] >= 15][["player", "team", "shots", "goals", "xg"]].head(20)
print(top.to_string(index=False))

Joueurs avec 15+ tirs sur la saison :
                                          player      team  shots  goals    xg
                  Lionel Andrés Messi Cuccittini Barcelona    195     30 22.87
                               Antoine Griezmann Barcelona     64     12 12.55
                                 Ousmane Dembélé Barcelona     47      5  4.54
                  Martin Braithwaite Christensen Barcelona     22      2  4.35
                                 Frenkie de Jong Barcelona     15      2  4.21
                       Philippe Coutinho Correia Barcelona     27      2  3.69
                            Pedro González López Barcelona     22      3  3.64
Francisco António Machado Mota de Castro Trincão Barcelona     20      3  2.39
                                Jordi Alba Ramos Barcelona     24      3  1.96


In [28]:
# Tableau comparatif face-à-face
try:
    comp = compare_players(shots, "Messi", "Griezmann")
    print(comp.to_string(index=False))
except ValueError as e:
    print(f"⚠️ {e}")
    print("Essaie avec un autre joueur du tableau ci-dessus.")

          métrique  Lionel Andrés Messi Cuccittini  Antoine Griezmann
             shots                         195.000             64.000
             goals                          30.000             12.000
                xg                          22.870             12.550
       xg_per_shot                           0.117              0.196
   conversion_rate                          15.400             18.800
   overperformance                           7.130             -0.550
   shots_on_target                          89.000             29.000
    on_target_rate                          45.600             45.300
       big_chances                          17.000             12.000
big_chances_scored                          10.000              7.000
      headed_shots                           5.000              8.000
   left_foot_shots                         172.000             41.000
  right_foot_shots                          17.000             15.000


---
## 11. Analyse par équipe

Comparons l'efficacité offensive de toutes les équipes de la saison.

In [29]:
# Profil de toutes les équipes
teams = team_shooting_profile(shots)
print("Profil offensif — toutes les équipes La Liga 2020/21")
print("="*80)
print(teams[[
    "team", "matches_played", "shots", "goals", "xg",
    "overperformance", "xg_per_match", "xg_per_shot", "big_chances"
]].to_string(index=False))

Profil offensif — toutes les équipes La Liga 2020/21
            team  matches_played  shots  goals    xg  overperformance  xg_per_match  xg_per_shot  big_chances
       Barcelona              35    543     76 72.02             3.98          2.06        0.133           62
     Real Madrid               2     30      5  4.22             0.78          2.11        0.141            2
           Cádiz               2     11      3  3.38            -0.38          1.69        0.307            3
          Getafe               2     14      2  3.15            -1.15          1.58        0.225            4
        Valencia               2     20      4  2.94             1.06          1.47        0.147            3
   Real Sociedad               2     22      2  2.61            -0.61          1.30        0.119            1
          Huesca               2     12      1  2.49            -1.49          1.25        0.207            2
      Real Betis               2     20      4  2.40             1.

In [30]:
# Graphique équipes : xG vs buts
fig = player_xg_bar(
    teams.rename(columns={"team": "player"}),   # réutilise la même fonction
    top_n=20,
    metric="xg",
    title="La Liga 2020/21 — xG vs Buts par équipe"
)
plt.show()

---
## 12. Conclusions & Insights

### 🔑 Ce que révèle l'analyse xG

**Sur les joueurs :**
- L'overperformance mesure l'efficacité NETTE du finisseur, indépendamment du volume
- Un xG/tir élevé indique un joueur qui crée des occasions de qualité (positions idéales)
- Un fort taux de cadrage révèle un tireur technique

**Sur les équipes :**
- `xg_per_match` est le meilleur indicateur de la menace offensive régulière
- L'overperformance d'équipe peut être signe d'un excellent gardien adverse, ou de variance

**Limites du modèle xG StatsBomb :**
- Le xG est calculé sur des critères spécifiques (angle, distance, technique...)
- Il ne capture pas tout : le niveau du gardien, la pression défensive, la fatigue
- Sur un faible échantillon (5 matchs), la variance est très élevée

---

### 🚀 Extensions possibles
- Intégrer les données de pressing pour contextualiser les tirs sous pression
- Analyser l'assist (key passes) pour remonter à l'origine des occasions
- Comparer avec la saison précédente (évolution d'un joueur)
- Intégrer tracking data (SkillCorner) pour les profils physiques

---
*Julie Landrevie — Football Data Analyst | StatsBomb Open Data | La Liga 2020/21*